# Chapter 5 — Build It Yourself: a GPT

Assemble the full Transformer. You'll build **multi-head attention**, the **feed-forward net**,
and the **residual + LayerNorm block** — then stack them into a GPT and train it on Shakespeare.
"✍️ Your turn", "📖 Study & run", "▶️ Check your work". (The notebook leaves out dropout for
clarity; `code/gpt.py` adds it.) 🚀

## Step 1 — Setup, data, and the attention Head  ▶️ Run this

Loads Shakespeare, sets the config, and defines `Head` — your Chapter 4 attention head, reused
unchanged.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

DATA = next(p for p in [Path("../data/input.txt"), Path("data/input.txt"),
                        Path("chapters/05-transformer/data/input.txt")] if p.exists())
text = DATA.read_text()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

torch.manual_seed(1337)
block_size, n_embd, n_head, n_layer, batch_size = 64, 128, 4, 4, 32

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    return (torch.stack([d[i:i+block_size] for i in ix]),
            torch.stack([d[i+1:i+block_size+1] for i in ix]))

class Head(nn.Module):                      # Chapter 4's head, unchanged
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)

print(f"{len(text)} chars | vocab {vocab_size} | {n_head} heads, {n_layer} layers")

## Step 2 — Multi-head attention  ✍️ Your turn

Run several heads in parallel and **concatenate** their outputs along the last dim (back to
`n_embd`), then **project** with `self.proj`.

<details><summary>Stuck? Show the answer</summary>

```python
out = torch.cat([h(x) for h in self.heads], dim=-1)
return self.proj(out)
```
</details>

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(head_size * n_head, n_embd)
    def forward(self, x):
        # ✍️ concatenate all heads' outputs along the last dim, then project
        out = x          # replace: combine the heads (cat, dim=-1)
        return out       # replace: project with self.proj

print("mha out:", tuple(MultiHeadAttention(4, n_embd // 4)(torch.randn(2, 8, n_embd)).shape))

In [ ]:
# ▶️ Check your work
try:
    mha = MultiHeadAttention(4, n_embd // 4)
    x = torch.randn(2, 8, n_embd)
    o = mha(x)
    assert tuple(o.shape) == (2, 8, n_embd), f"output should be (2, 8, {n_embd}), got {tuple(o.shape)}"
    assert len(mha.heads) == 4, "should have 4 heads"
    bare_cat = torch.cat([h(x) for h in mha.heads], dim=-1)
    assert not torch.allclose(o, bare_cat, atol=1e-6), "you returned the bare concatenation — apply self.proj to it"
    assert torch.allclose(o, mha.proj(bare_cat), atol=1e-6), "output should be self.proj(the concatenated heads)"
    print(f"✅ Multi-head attention works — 4 heads concatenated AND projected back to {n_embd}.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 3 — The feed-forward net  ✍️ Your turn

A per-position MLP: a `Linear` widening to `4 * n_embd`, a `ReLU`, then a `Linear` back to
`n_embd`. Use `nn.Sequential` to chain them.

<details><summary>Stuck? Show the answer</summary>

```python
self.net = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.ReLU(),
    nn.Linear(4 * n_embd, n_embd),
)
```
</details>

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        # ✍️ a Sequential MLP: widen to 4×, ReLU, then back to n_embd
        self.net = nn.Identity()     # replace with the real Sequential
    def forward(self, x):
        return self.net(x)

print("ffwd out:", tuple(FeedForward(n_embd)(torch.randn(2, 8, n_embd)).shape))

In [ ]:
# ▶️ Check your work
try:
    ff = FeedForward(n_embd)
    o = ff(torch.randn(2, 8, n_embd))
    assert tuple(o.shape) == (2, 8, n_embd), f"output should be (2, 8, {n_embd}), got {tuple(o.shape)}"
    assert any(isinstance(m, nn.ReLU) for m in ff.net.modules()), "the net needs a ReLU nonlinearity in the middle"
    print(f"✅ Feed-forward net works — widens to {4*n_embd}, ReLU, back to {n_embd}.")
except (NameError, AssertionError, AttributeError, TypeError) as e:
    print("❌", e)

## Step 4 — The Transformer block 🔑  ✍️ Your turn

The key idea: **residual + pre-norm**. Add each sublayer's output back to `x` (the residual),
and LayerNorm `x` *before* feeding each sublayer. Fill the two `forward` lines.

<details><summary>Stuck? Show the answer</summary>

```python
x = x + self.sa(self.ln1(x))      # attention sublayer, added back
x = x + self.ffwd(self.ln2(x))    # feed-forward sublayer, added back
```
</details>

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        # ✍️ residual + pre-norm: ADD each sublayer's output (computed on the LayerNormed x) back to x
        x = self.sa(self.ln1(x))      # ← add this to x
        x = self.ffwd(self.ln2(x))    # ← add this to x
        return x

print("block out:", tuple(Block(n_embd, 4)(torch.randn(2, 8, n_embd)).shape))

In [ ]:
# ▶️ Check your work (recomputes the correct residual forward and compares)
try:
    torch.manual_seed(0)
    blk = Block(n_embd, 4)
    xin = torch.randn(2, 8, n_embd)
    out = blk(xin)
    h = xin + blk.sa(blk.ln1(xin))           # the correct residual forward, recomputed
    expected = h + blk.ffwd(blk.ln2(h))
    assert tuple(out.shape) == (2, 8, n_embd), f"shape should be (2, 8, {n_embd}), got {tuple(out.shape)}"
    assert torch.allclose(out, expected, atol=1e-5), "the block must ADD each sublayer back to x (residual): x = x + sa(ln1(x)); x = x + ffwd(ln2(x))"
    print("✅ Transformer block works — residual + pre-norm, the key to trainable depth.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 5 — Assemble the GPT  📖 Study & run

Token + position embeddings → a stack of `n_layer` blocks → a final LayerNorm → a `Linear` to
`vocab_size`. This is a decoder-only Transformer — architecturally, GPT-2.

In [ ]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
        return idx

model = GPT()
print(f"{sum(p.nelement() for p in model.parameters())/1e3:.0f}K parameters")

## Step 6 — Train it and read the result  📖 Study & run

~2000 steps (about a minute). Watch the val loss fall well below Chapter 4's 2.34, then read
the sample — recognizable, if broken, English.

In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=30):
    out = {}; model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            losses[k] = model(*get_batch(split))[1].item()
        out[split] = losses.mean().item()
    model.train(); return out

opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
for step_i in range(2000):
    _, loss = model(*get_batch("train"))
    opt.zero_grad(); loss.backward(); opt.step()
    if step_i % 500 == 0:
        L = estimate_loss()
        print(f"step {step_i:4d} | train {L['train']:.4f} | val {L['val']:.4f}")

val_loss = estimate_loss()["val"]
print(f"\nfinal val {val_loss:.4f}  (Chapter 4's single head was 2.34)\n--- sample ---")
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long), 400)[0].tolist()))

In [ ]:
# ▶️ Check your work
try:
    assert val_loss < 2.1, f"val should beat Chapter 4's 2.34, got {val_loss:.4f}"
    print(f"✅ Your GPT trained to val {val_loss:.4f} — well below Chapter 4's 2.34. You built a Transformer!")
except (NameError, AssertionError) as e:
    print("❌", e)

## 🎓 You built a GPT.

Multi-head attention, a feed-forward net, residual connections, LayerNorm — stacked into blocks,
stacked into a decoder-only Transformer. It's the architecture of GPT-2 and ChatGPT; yours just
needs scale. The text is broken because the model is tiny and reads *characters* — Chapter 6
(tokenization) and the speed chapters fix exactly that.

Next: the [exercises](../exercises/) (ablate the residuals and watch it break!) and the
[mini-project](../project/), *Your First GPT*. 👋